# 03 — Data Wrangling & Feature Engineering
**IBM Applied Data Science Capstone — Harshpreet Singh**

GitHub: https://github.com/harshps1001/ds-capstone-spacex

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/spacex_launch_dash.csv')
print(f'Shape: {df.shape}')
print(df.dtypes)

## Filter to Falcon 9 Only

In [ ]:
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].reset_index(drop=True)
print(f'After filtering: {data_falcon9.shape}')

After filtering: (90, 17)


## Handle Missing Values

In [ ]:
print('Missing values per column:')
print(data_falcon9.isnull().sum()[data_falcon9.isnull().sum() > 0])

# Impute PayloadMass with column mean
data_falcon9['PayloadMass'].fillna(data_falcon9['PayloadMass'].mean(), inplace=True)
print(f'After imputation: {data_falcon9["PayloadMass"].isnull().sum()} nulls')

Missing values per column:
PayloadMass    5
dtype: int64
After imputation: 0 nulls


## Create Binary Target Variable (Class)

In [ ]:
landing_outcomes = data_falcon9['Outcome'].value_counts()
print('Landing outcomes:')
print(landing_outcomes)

bad_outcomes = {'False Ocean','False RTLS','False ASDS','None ASDS','None None'}
data_falcon9['Class'] = data_falcon9['Outcome'].apply(lambda x: 0 if x in bad_outcomes else 1)
print(f'Class distribution: {dict(data_falcon9["Class"].value_counts().sort_index())}')

Landing outcomes:
True ASDS    41
None None    21
False ASDS    6
True RTLS    14
False Ocean    2
False RTLS    1
None ASDS    4
dtype: int64
Class distribution: {0: 33, 1: 57}


## One-Hot Encoding & Train/Test Split

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

features = pd.get_dummies(data_falcon9[['PayloadMass','Orbit','LaunchSite','Flights',
                                         'GridFins','Reused','Legs','LandingPad',
                                         'Block','ReusedCount','Serial']])
print(f'Feature matrix shape: {features.shape}')

X = features.values
y = data_falcon9['Class'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

Feature matrix shape: (90, 83)
Train: 72 | Test: 18
